<a href="https://colab.research.google.com/github/cliteka-cell/ner-wikiann/blob/main/exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate accelerate seqeval

from datasets import load_dataset

dataset = load_dataset('wikiann', 'en')
print(dataset)

In [ ]:
label_names = dataset['train'].features['ner_tags'].feature.names
print('Labels:', label_names)

In [ ]:
example = dataset['train'][0]

for token, tag in zip(example['tokens'], example['ner_tags']):
    print(f"{token:20} {label_names[tag]}")

In [ ]:
from collections import Counter

all_tags = [tag for row in dataset['train']['ner_tags'] for tag in row]
counts = Counter(all_tags)

print(f"{'Label':12} {'Count':>8}  {'%':>6}")
print('-' * 30)
for tag_id, count in sorted(counts.items()):
    print(f"{label_names[tag_id]:12} {count:>8}  {count/len(all_tags)*100:>5.1f}%")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != prev:
                aligned.append(labels[word_id])
            else:
                aligned.append(-100)
            prev = word_id
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)
print("Done!", tokenized_dataset)

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names),
    id2label={i: l for i, l in enumerate(label_names)},
    label2id={l: i for i, l in enumerate(label_names)}
)

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [label_names[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {"f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

args = TrainingArguments(
    output_dir="ner-results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(f"F1 Score:  {results['eval_f1']:.4f}")
print(f"Accuracy:  {results['eval_accuracy']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

In [ ]:
from transformers import pipeline

ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

sentence = "Abraham Lincoln was born in Larue County, KY and Lincoln Memorial in Washington D.C. was built for him by Henry Bacon."
results_ner = ner(sentence)

for entity in results_ner:
    print(f"{entity['word']:20} {entity['entity_group']:8} {entity['score']:.2f}")

In [ ]:
# Run model on the full test set
predictions = trainer.predict(tokenized_dataset["test"])
pred_labels = predictions.predictions.argmax(-1)
true_labels = predictions.label_ids

print("Predictions ready!")
print(f"Test set size: {len(pred_labels)} sentences")

In [ ]:
errors = {"wrong_type": [], "false_positive": [], "false_negative": []}

for sent_idx in range(len(true_labels)):
    tokens = dataset["test"][sent_idx]["tokens"]
    true = [label_names[l] for l in true_labels[sent_idx] if l != -100]
    pred = [label_names[p] for p, l in zip(pred_labels[sent_idx], true_labels[sent_idx]) if l != -100]

    for i, (t, p) in enumerate(zip(true, pred)):
        token = tokens[i] if i < len(tokens) else "?"
        if t == p:
            continue
        if t == "O" and p != "O":
            errors["false_positive"].append({"token": token, "predicted": p})
        elif t != "O" and p == "O":
            errors["false_negative"].append({"token": token, "true": t})
        elif t != "O" and p != "O" and t != p:
            errors["wrong_type"].append({"token": token, "true": t, "predicted": p})

print(f"Wrong type:      {len(errors['wrong_type'])}")
print(f"False positives: {len(errors['false_positive'])}")
print(f"False negatives: {len(errors['false_negative'])}")

In [ ]:
from collections import Counter

# Most common wrong type confusions
confusions = Counter(
    (e["true"], e["predicted"]) for e in errors["wrong_type"]
)

print(f"{'True':12} {'Predicted':12} {'Count':>6}")
print("-" * 32)
for (true, pred), count in confusions.most_common(10):
    print(f"{true:12} {pred:12} {count:>6}")

In [ ]:
# Find real sentence examples for top confusions
def find_examples(true_label, pred_label, n=3):
    examples = []
    for sent_idx in range(len(true_labels)):
        tokens = dataset["test"][sent_idx]["tokens"]
        true = [label_names[l] for l in true_labels[sent_idx] if l != -100]
        pred = [label_names[p] for p, l in zip(pred_labels[sent_idx], true_labels[sent_idx]) if l != -100]
        for i, (t, p) in enumerate(zip(true, pred)):
            if t == true_label and p == pred_label and i < len(tokens):
                examples.append({
                    "token": tokens[i],
                    "sentence": " ".join(tokens)
                })
        if len(examples) >= n:
            break
    return examples

# Show examples for top 3 confusions
for true_label, pred_label in [("I-LOC", "I-ORG"), ("I-PER", "I-ORG"), ("B-ORG", "B-LOC")]:
    print(f"\n{true_label} → {pred_label}:")
    for ex in find_examples(true_label, pred_label):
        print(f"  '{ex['token']}' in: {ex['sentence'][:80]}")

In [ ]:
print("=== ERROR ANALYSIS SUMMARY ===\n")
print(f"Total test tokens evaluated: {sum(len(s) for s in dataset['test']['tokens'])}")
print(f"Wrong type errors:  {len(errors['wrong_type'])} ({len(errors['wrong_type'])/(len(errors['wrong_type'])+len(errors['false_positive'])+len(errors['false_negative']))*100:.1f}% of all errors)")
print(f"False positives:    {len(errors['false_positive'])} ({len(errors['false_positive'])/(len(errors['wrong_type'])+len(errors['false_positive'])+len(errors['false_negative']))*100:.1f}% of all errors)")
print(f"False negatives:    {len(errors['false_negative'])} ({len(errors['false_negative'])/(len(errors['wrong_type'])+len(errors['false_positive'])+len(errors['false_negative']))*100:.1f}% of all errors)")
print(f"\nTop confusion: LOC↔ORG ({685+594} cases) — geopolitical entity ambiguity")
print(f"Dataset errors found: model predictions arguably more correct than ground truth labels in some cases")

In [ ]:
from google.colab import output
output.clear()